# TF-IDF 特征构建 — 每条评论一个 TF-IDF 向量

**数据**: `Data/Human/Human_Result.csv`（423 条，人工标注为黄金标准）
- `负面性标签` / `侮辱性标签`：二分类标签，层级约束「侮辱性 ⊆ 负面性」（已验证 0 条违反）

## 建模路线总览

| 路线 | 输入特征 | 说明 |
|---|---|---|
| 传统 ML（SVM / NB / LR） | **本 notebook 的 TF-IDF 矩阵** | 稀疏高维，TF-IDF 是标准输入 |
| 深度学习（TextCNN / LSTM） | 词 ID 序列 / 词向量 | **不需要 TF-IDF**，见文末说明 |
| LLM zero-shot / 提示词工程 | 原始评论文本 | 不需要特征工程 |

## 需要准备的东西（本 notebook 的前置清单）

1. **中文分词器**：jieba（已安装）
2. **侮辱性词典**：进阶 TODO:
3. **中文停用词表**：常规虚词（的、了、是…）过滤；**注意不要过滤否定词**（不、没、别），「不好吃」和「好吃」差一个 Negation 就是两个标签
4. **Emoji 策略**：本语料 emoji 带情感信号（🤣😭😋），**保留**，作为独立 token
5. **数据划分意识**：目前对全量拟合 TF-IDF 用于探索；正式建模时 `TfidfVectorizer` **只能在训练集上 fit**，再 transform 测试集，否则标签信息泄漏、指标虚高

In [1]:
from pathlib import Path

import jieba
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

DATA_PATH = "../Data/Human/Human_Result.csv"
OUT_DIR = "../Data/ML/"

# 否定词必须保留，不能进停用词表
NEGATION_WORDS = {"不", "没", "没有", "别", "莫", "勿", "未", "无"}

# 内置最小停用词表（若未提供外部停用词表则使用）
FALLBACK_STOPWORDS = {
    "的", "了", "是", "我", "你", "他", "她", "它", "我们", "你们", "这", "那",
    "也", "都", "就", "在", "有", "人", "啊", "吗", "呢", "吧", "呀", "哦",
    "和", "跟", "与", "或", "而且", "但是", "所以", "因为", "如果", "虽然",
    "一个", "一下", "还是", "这个", "那个", "什么", "怎么", "这么", "那么",
}


In [2]:
stopwords = set(FALLBACK_STOPWORDS)
stopwords -= NEGATION_WORDS   # 否定词绝不停用
print("停用词表中剔除的否定词:", NEGATION_WORDS & stopwords or "无")

停用词表中剔除的否定词: 无


In [3]:
df = pd.read_csv(DATA_PATH)
print(f"样本数: {len(df)}，标签分布: 负面性={df['负面性标签'].sum()}，侮辱性={df['侮辱性标签'].sum()}")
assert ((df['侮辱性标签'] == 1) & (df['负面性标签'] == 0)).sum() == 0, "层级约束被违反"

# 注意：有 15 条重复评论，建模划分时要用 GroupShuffleSplit 防止同一条评论同时进训练集和测试集
print(f"重复评论条数: {df['评论'].duplicated().sum()}")

样本数: 423，标签分布: 负面性=237，侮辱性=45
重复评论条数: 15


## 分词与预处理

- jieba 分词（侮辱词典已作为自定义词典加载，确保骂人短语不被切碎）
- 过滤空白、标点、纯数字；保留 emoji（作为独立 token，它带情感信号）
- 每条评论 → 一串 token，TF-IDF 以"每条评论"为一个文档

In [4]:
import re

EMOJI_RE = re.compile(
    "[\U0001F300-\U0001FAFF\U00002700-\U000027BF\U0001F000-\U0001F02F"
    "\U00002600-\U000026FF\U0001F900-\U0001F9FF\U00002764\U0001F525]"
)

def tokenize(text: str) -> list[str]:
    """中文分词 + emoji 独立成 token + 过滤停用词/标点/纯数字。"""
    tokens = []
    for tok in EMOJI_RE.findall(text):
        tokens.append(tok)              # emoji 保留
    text_no_emoji = EMOJI_RE.sub(" ", text)
    for tok in jieba.lcut(text_no_emoji):
        tok = tok.strip()
        if not tok or tok in stopwords:      # 空串、停用词
            continue
        if tok.isdigit():                    # 纯数字
            continue
        if re.fullmatch(r"[\W_]+", tok, re.UNICODE):   # 纯标点
            continue
        tokens.append(tok)
    return tokens

df["tokens"] = df["评论"].apply(tokenize)
df["token_text"] = df["tokens"].apply(" ".join)

print("分词完成，示例：")
for i in range(3):
    print(f"  [{df.loc[i, '类型']}] {df.loc[i, '评论'][:30]}...")
    print(f"    -> {df.loc[i, 'tokens'][:15]}")
print(f"平均 token 数: {df['tokens'].apply(len).mean():.1f}")

Building prefix dict from the default dictionary ...
Loading model from cache /var/folders/yt/b96z1pj11z35bffvldp7796c0000gn/T/jieba.cache
Loading model cost 0.409 seconds.
Prefix dict has been built successfully.


分词完成，示例：
  [语气型差评] 用料扎实到咬不动 牙医听了都连夜谢谢你们家🤣...
    -> ['🤣', '用料', '扎实', '到', '咬', '不动', '牙医', '听', '连夜', '谢谢你们', '家']
  [语气型差评] 菜里面竟然附赠了一条🐛蠕动的赠品，真是太贴心了😭...
    -> ['🐛', '😭', '菜', '里面', '竟然', '附赠', '一条', '蠕动', '赠品', '真是太', '贴心']
  [语气型差评] 老板仁义啊！居然拿半个世纪前的古董来做菜！菜叶枯黄得像秋天的...
    -> ['😋', '老板', '仁义', '居然', '拿', '半个世纪', '前', '古董', '来', '做菜', '菜叶', '枯黄', '得', '像', '秋天']
平均 token 数: 12.7


## TF-IDF 计算

**怎么算的**：`TfidfVectorizer` 把每条评论当一个文档，输出 N×V 稀疏矩阵（N=423 条评论，V=词表大小），每一**行**就是该评论的 TF-IDF 向量。

- `token_pattern=None`：因为已经自己分好词，按空格切即可
- `ngram_range=(1, 2)`：保留二元短语（"难吃 到 窒息"这类搭配对情感判别很有用）
- `sublinear_tf=True`：词频做 1+log(tf) 压缩，长评论不会被少数高频词主导
- `min_df=2`：至少出现在 2 条评论里，去掉孤例噪声

In [5]:
vectorizer = TfidfVectorizer(
    analyzer="word",
    token_pattern=None,          # 输入已是空格分隔的 token
    tokenizer=str.split,
    preprocessor=None,
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)

X_tfidf = vectorizer.fit_transform(df["token_text"])
print(f"TF-IDF 矩阵形状: {X_tfidf.shape}（N x V，稀疏度 {(1 - X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])):.2%}）")

vocab = np.array(vectorizer.get_feature_names_out())
print(f"词表大小 V = {len(vocab)}")

TF-IDF 矩阵形状: (423, 991)（N x V，稀疏度 98.81%）
词表大小 V = 991


In [6]:
# 看看词表里都有些什么：按出现次数列出最高的 20 个 token
from collections import Counter
term_freq = Counter()
for toks in df["tokens"]:
    term_freq.update(toks)
print("出现次数最高的 20 个 token:")
for w, c in term_freq.most_common(20):
    print(f"  {w}: {c}")

出现次数最高的 20 个 token:
  好吃: 131
  很: 107
  吃: 94
  味道: 92
  饭: 71
  还: 62
  木桶: 60
  好: 54
  点: 45
  不错: 45
  不: 42
  好好: 38
  这家: 36
  肉: 33
  外卖: 32
  真的: 32
  非常: 31
  米饭: 31
  可以: 29
  推荐: 29


In [7]:
# 保存：TF-IDF 稀疏矩阵 + 词表 + 带 token 列的数据表
import scipy.sparse as sp

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
sp.save_npz(OUT_DIR / "tfidf_matrix.npz", X_tfidf)
with open(OUT_DIR / "tfidf_vocab.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(vocab))
df.drop(columns=["tokens"]).to_csv(OUT_DIR / "human_tokenized.csv", index=False, encoding="utf-8-sig")

print("已保存:")
print(f"  {OUT_DIR / 'tfidf_matrix.npz'}   （稀疏矩阵，行序与 Human_Result.csv 一致）")
print(f"  {OUT_DIR / 'tfidf_vocab.txt'}  （词表，第 j 行对应矩阵第 j 列）")
print(f"  {OUT_DIR / 'human_tokenized.csv'} （含 token_text 列，供复现）")

已保存:
  ../Data/ML/tfidf_matrix.npz   （稀疏矩阵，行序与 Human_Result.csv 一致）
  ../Data/ML/tfidf_vocab.txt  （词表，第 j 行对应矩阵第 j 列）
  ../Data/ML/human_tokenized.csv （含 token_text 列，供复现）


## ML 分类：区分负面性与侮辱性

**设计要点：为什么用交叉验证（CV）而不是"全量训练 + 全量预测"**

如果把全部 423 条喂给模型再让它预测这 423 条，模型是在"自己考自己"——见过的样本预测准确率必然虚高，和人工标注的对比没有意义。因此这里用 **分层 5 折交叉验证（Stratified 5-Fold CV）** 收集 **out-of-fold (OOF) 预测**：每条评论的预测都来自一个**没见过这条评论**的模型，可以公平地与人工黄金标注比较。

- **分层**：侮辱性正例仅 45/423（10.6%），每折按标签比例抽样，保证每折都有正例
- **防泄漏**：TF-IDF 也放进 Pipeline 里随每折重新 fit——验证折的词表不掺入训练信息
- **算法**：逻辑回归 / 朴素贝叶斯 / 线性SVM / 随机森林，四个经典 ML 基线

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# 用上一节保存的分词结果（token_text 列）
df_ml = pd.read_csv("../Data/ML/human_tokenized.csv")
print(f"样本数: {len(df_ml)}")
print(f"负面性正例: {df_ml['负面性标签'].sum()}，侮辱性正例: {df_ml['侮辱性标签'].sum()}")

样本数: 423
负面性正例: 237，侮辱性正例: 45


In [9]:
# 与前文 TF-IDF 参数完全一致（放进 Pipeline 随折拟合，防泄漏）
def make_tfidf():
    return TfidfVectorizer(
        analyzer="word", token_pattern=None, tokenizer=str.split,
        lowercase=False, ngram_range=(1, 2), min_df=2, sublinear_tf=True,
    )

ALGOS = {
    "逻辑回归": lambda: LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    "朴素贝叶斯": lambda: MultinomialNB(),
    "线性SVM": lambda: LinearSVC(class_weight="balanced", random_state=42),
    "随机森林": lambda: RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def oof_predict(algo_name, y):
    """分层5折CV，返回全体样本的 out-of-fold 预测。"""
    pipe = Pipeline([("tfidf", make_tfidf()), ("clf", ALGOS[algo_name]())])
    return cross_val_predict(pipe, df_ml["token_text"].fillna(""), y, cv=cv, n_jobs=-1)

print("算法与CV方案就绪:", list(ALGOS), "| 5折分层")

算法与CV方案就绪: ['逻辑回归', '朴素贝叶斯', '线性SVM', '随机森林'] | 5折分层


In [10]:
DIMS = ["负面性", "侮辱性"]
oof = {}        # (algo, dim) -> 预测数组
metrics = []    # 评估结果行

for dim in DIMS:
    y = df_ml[f"{dim}标签"].to_numpy()
    for algo in ALGOS:
        pred = oof_predict(algo, y)
        oof[(algo, dim)] = pred
        metrics.append({
            "维度": dim, "算法": algo,
            "Accuracy": accuracy_score(y, pred),
            "Precision": precision_score(y, pred, zero_division=0),
            "Recall": recall_score(y, pred, zero_division=0),
            "F1": f1_score(y, pred, zero_division=0),
        })

metrics_df = pd.DataFrame(metrics).round(4)
for dim in DIMS:
    print(f"===== {dim}（OOF 5折CV） =====")
    print(metrics_df[metrics_df['维度'] == dim].drop(columns='维度').to_string(index=False))
    print()

===== 负面性（OOF 5折CV） =====
   算法  Accuracy  Precision  Recall     F1
 逻辑回归    0.8913     0.8805  0.9325 0.9057
朴素贝叶斯    0.8865     0.8765  0.9283 0.9016
线性SVM    0.9031     0.9083  0.9198 0.9140
 随机森林    0.8818     0.8696  0.9283 0.8980

===== 侮辱性（OOF 5折CV） =====
   算法  Accuracy  Precision  Recall     F1
 逻辑回归    0.8392     0.2553  0.2667 0.2609
朴素贝叶斯    0.8936     0.0000  0.0000 0.0000
线性SVM    0.8440     0.2000  0.1556 0.1750
 随机森林    0.8794     0.2000  0.0444 0.0727



### 层级约束后处理

业务规则「侮辱性 ⊆ 负面性」（骂商家一定是在给差评）：若模型预测侮辱=1 但负面=0，则逻辑矛盾。处理方式：**把该条的负面性预测修正为 1**（预测了侮辱，就意味着给出了差评）。修正后的负面性列单独命名，与原始预测并列，便于对比。

In [11]:
# 层级修正：侮辱=1 蕴含 负面=1
corrections = {}
for algo in ALGOS:
    neg, ins = oof[(algo, "负面性")].copy(), oof[(algo, "侮辱性")]
    n_fixed = int(((ins == 1) & (neg == 0)).sum())
    neg[(ins == 1) & (neg == 0)] = 1
    oof[(algo, "负面性_层级修正")] = neg
    corrections[algo] = n_fixed
print("各算法被层级修正的负面性预测条数:", corrections)

各算法被层级修正的负面性预测条数: {'逻辑回归': 0, '朴素贝叶斯': 0, '线性SVM': 2, '随机森林': 0}


In [12]:
out = df_ml[["类型", "负面性标签", "侮辱性标签", "评论"]].copy()

for algo in ALGOS:
    out[f"{algo}+负面性"] = oof[(algo, "负面性_层级修正")]   # 负面性用层级修正后的结果
    out[f"{algo}+侮辱性"] = oof[(algo, "侮辱性")]

out_path = "../Data/ML/ML_predictions.csv"
out.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"已保存: {out_path}")
print(f"形状: {out.shape}（423 行 x [标签4列 + {len(ALGOS)}算法 x 2 维度] 列）")
out.head()

已保存: ../Data/ML/ML_predictions.csv
形状: (423, 12)（423 行 x [标签4列 + 4算法 x 2 维度] 列）


,类型,负面性标签,侮辱性标签,评论,逻辑回归+负面性,逻辑回归+侮辱性,朴素贝叶斯+负面性,朴素贝叶斯+侮辱性,线性SVM+负面性,线性SVM+侮辱性,随机森林+负面性,随机森林+侮辱性
0,语气型差评,1,0,用料扎实到咬不动 牙医听了都连夜谢谢你们家🤣,1,0,1,0,1,0,1,0
1,语气型差评,1,0,菜里面竟然附赠了一条🐛蠕动的赠品，真是太贴心了😭,1,0,1,0,1,0,1,1
2,语气型差评,1,0,老板仁义啊！居然拿半个世纪前的古董来做菜！菜叶枯黄得像秋天的落叶，不得不分五期给五星好评了捏😋,1,0,1,0,1,0,1,0
3,语气型差评,1,0,点的微辣，结果辣得我全家连夜去医院挂肛肠科，真是一场酣畅淋漓的体验。,1,0,1,0,1,0,1,0
4,语气型差评,1,0,这菜的味道极其富有层次感，先是齁咸，接着是苦涩，最后是一股刷锅水味，厨师真是个调味鬼才,1,0,1,0,1,0,1,0


### 结果解读要点

- **看 F1 不看 Accuracy**：侮辱性正例只占 10.6%，全预测 0 也有 89.4% accuracy，毫无意义；`class_weight="balanced"` 已在算法侧缓解不平衡，评估以 F1 为准
- **预期排序**：线性模型（SVM / 逻辑回归）通常在小型中文 TF-IDF 任务上最好；随机森林在稀疏高维文本上一般最弱，作为对照保留
- **与 LLM 对比时的公平性**：ML 这边每条评论的预测都来自未见过的模型（OOF），LLM 侧也应保证同样的"零样本接触"——不能把人工标注喂给 LLM 再让它预测同样数据
- 负面性列已是**层级修正后**的结果（侮辱⇒负面），若想要原始未修正预测，把 `负面性_层级修正` 换成 `负面性` 即可